In [6]:
del model
del generator

In [1]:

from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import warnings
warnings.filterwarnings('ignore')

In [6]:

# Checking current setup
import torch
import platform
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Platform: {platform.platform()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

Python version: 3.10.14 (main, Aug 14 2024, 05:14:46) [Clang 18.1.8 ]
PyTorch version: 2.3.1
Platform: macOS-26.1-arm64-arm-64bit
MPS available: True
MPS built: True


In [3]:

import os

# NO NEED FOR THIS ANYMORE, GIVEN THE ABOVE
# os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Force MPS availability check
# if hasattr(torch.backends, 'mps'):
    # torch.backends.mps.is_available = lambda: True

# print(f"MPS available after override: {torch.backends.mps.is_available()}")

In [37]:

# DOING THIS [FAST CPU] SINCE MPS TEMPORARILY UNAVAILABLE

def setup_fast_cpu_pipeline():
    """Setup a fast pipeline optimized for CPU"""
    
    # These work well on CPU and are fast enough
    model_options = [
        "distilgpt2",           # Fastest
        "microsoft/DialoGPT-medium",  # Good for conversational tasks
        "gpt2",                 # Reliable
    ]
    
    generator = pipeline(
        "text-generation",
        model=model_options[2],  # Start with fastest
        device="cpu",
        torch_dtype=torch.float32
    )
    
    return generator

In [38]:

generator = setup_fast_cpu_pipeline()

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


In [25]:

def pre_clean(entry_list):
    """
    Filters out OCR junk and non-name strings before sending to LLM.
    Keeps only plausible person-like strings.
    """
    cleaned = []
    name_pattern = re.compile(r"^[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑ]?[a-záéíóúñ]+)*$")
    short_name_pattern = re.compile(r"^[A-ZÁÉÍÓÚÑ][a-záéíóúñ]{2,}$")

    for e in entry_list:
        name = e.get("author_name", "").strip()
        if not name:
            continue
        # Remove junk: too short, digits, symbols, or excessive punctuation
        if len(name) < 3 or len(name.split()) > 8:
            continue
        if any(ch.isdigit() for ch in name):
            continue
        # if re.search(r"[@#/{}<>\[\]|]", name):
        if re.search(r"\d|[@#%&<>/\\]", name):
            continue
        if name.count(".") > 1:
            continue
        # Allow single-word proper names or multiword likely names
        if name_pattern.match(name) or short_name_pattern.match(name):
            cleaned.append(e)
    return cleaned


In [27]:

BATCH_SIZE = 10
AUTOSAVE_EVERY = 500  # Save progress every N entries
OUTPUT_FILE = "ahira_cmola_cleaned_toc_authors_by_distilgpt2.json"
BACKUP_PICKLE = True
RESUME = True
TEMPERATURE = 0.2
MAX_NEW_TOKENS = 256
SLEEP_BETWEEN_BATCHES = 0.2

In [60]:

import json, os, re, time, pickle
from tqdm import tqdm
import difflib, unicodedata

# --------------------------------
# 🔠 Normalization + Author List + Fuzzy Filter
# --------------------------------

def normalize_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    return ''.join(
        c for c in unicodedata.normalize('NFD', name)
        if unicodedata.category(c) != 'Mn'
    ).strip().lower()


TIER1_CLASSICS = {
    # Ancient Greek & Hellenistic
    "homero", "hesiodo", "safo", "arquiloco", "pindaro", "esquilo", "sofocles", "euripides",
    "herodoto", "tucides", "xenofonte", "platon", "aristoteles", "demostenes", "epicuro",
    "empedocles", "parmenides", "anaximandro", "anaximenes", "heraclito", "tales",
    "pitagoras", "socrates", "hipocrates", "euclides", "arquimedes", "galeno",

    # Roman / Latin
    "virgilio", "horacio", "ovidio", "catulo", "propecio", "tibulo", "juvenal", "persio",
    "lucrecio", "ciceron", "seneca", "tacito", "suetonio", "petronio", "apuleyo",
    "quintiliano", "terencio", "plauto", "marcial", "vitruvio",

    # Middle Eastern / Islamic Golden Age
    "avicenna", "averroes", "alfarabi", "alhazen", "rumi", "hafiz", "omarjayyam",
    "jayyam", "ibnkhaldun", "ibnsina", "algazali", "aljahiz", "alkindi",

    # Eastern / Asian
    "confucio", "laozi", "laotse", "mencio", "zhuangzi", "sunzi", "suntzu", "basho",
    "saigyo", "dogen", "libai", "dufu", "wangwei", "kalidasa",

    # Medieval & Renaissance
    "agustin", "tomas", "anselmo", "dante", "petrarca", "boccaccio", "erasmo",
    "maquiavelo", "montaigne", "moro", "chaucer", "villon", "tasso", "ariosto",

    # Early Modern / Enlightenment
    "cervantes", "shakespeare", "moliere", "voltaire", "rousseau", "montesquieu",
    "diderot", "pascal", "descartes", "newton", "leibniz", "spinoza", "hobbes",
    "locke", "kant", "hume",

    # 19th century core
    "goethe", "schiller", "byron", "shelley", "keats", "wordsworth", "coleridge",
    "tennyson", "whitman", "poe", "emerson", "thoreau", "balzac", "flaubert",
    "stendhal", "zola", "hugo", "tolstoi", "dostoievski", "pushkin", "nietzsche",
    "marx", "darwin", "freud", "kierkegaard", "schopenhauer", "baudelaire",
    "rimbaud", "mallarme", "wilde", "ibsen", "strindberg", "twain", "melville",
    "dickens",

    # 20th-century universally iconic
    "joyce", "proust", "kafka", "borges", "lorca", "neruda", "paz", "woolf",
    "beckett", "sartre", "camus", "einstein", "picasso", "beethoven", "bach",
    "mozart", "michelangelo", "galileo"
}


# -------------------
# 📚 Tier 2: Extended “Likely Surname-Only” Authors / Thinkers / Artists
# -------------------

TIER2_EXTENDED = {
    "rilke", "valery", "mistral", "dario", "pessoa", "benedetti", "machado",
    "baroja", "unamuno", "azorin", "galdos", "rodo", "marti", "bioy",
    "cortazar", "sabato", "onetti", "carpentier", "guillen", "hernandez",
    "miguelhernandez", "vallejo", "saramago", "vicente", "huidobro",
    "gongora", "quevedo", "garcilaso", "dali", "velazquez", "goya",
    "leonardo", "rafael", "delacroix", "rodin", "kandinsky", "chopin",
    "wagner", "debussy", "verdi", "mahler", "strauss", "wittgenstein",
    "heidegger", "barthes", "foucault", "derrida", "lacan", "arendt",
    "jung", "piaget", "saussure", "chomsky", "simone", "sontag", "orwell",
    "hemingway", "faulkner", "fitzgerald", "carver", "miller", "beckett",
    "celan", "cocteau", "breton", "eliot", "auden", "plath", "bellow",
    "nabokov", "solzhenitsyn", "eco", "calvino", "fuentes", "rulfo",
    "garcia", "marquez", "benjamin", "adorno", "horkheimer"
}


STRICT_MODE = False
CLASSIC_ONE_NAME_AUTHORS = {
    normalize_name(n)
    for n in (TIER1_CLASSICS if STRICT_MODE else TIER1_CLASSICS | TIER2_EXTENDED)
}

def is_known_classic_fuzzy(name: str) -> bool:
    """Allow fuzzy OCR-tolerant match to classic one-name authors."""
    norm = normalize_name(name)
    if norm in CLASSIC_ONE_NAME_AUTHORS:
        return True
    matches = difflib.get_close_matches(norm, CLASSIC_ONE_NAME_AUTHORS, n=1, cutoff=0.8)
    return bool(matches)

def filter_one_word_names(name_list):
    """Filters one-word names, keeping only known classics or multi-word names."""
    clean = []
    skipped = []

    for n in name_list:
        if not isinstance(n, str):
            skipped.append(n)
            continue

        parts = n.split()
        if len(parts) > 1 or is_known_classic_fuzzy(n):
            clean.append(n)
        else:
            skipped.append(n)

    if skipped:
        print(f"⚠️ Skipped {len(skipped)} non-string or invalid names: {skipped[:5]}")
    return clean


In [56]:

def clean_author_names_with_checkpoint(results, output_file=OUTPUT_FILE):
    """
    Cleans author names with LLM, uses checkpoints and autosaves.
    Adds OCR-fuzzy mononymic-author filtering and retry on malformed JSON.
    """
    cleaned, processed = [], 0
    if os.path.exists(output_file):
        print(f"🔄 Resuming from {output_file}")
        with open(output_file, "r", encoding="utf-8") as f:
            cleaned = json.load(f)
        processed = len(cleaned)
        print(f"Resumed from {processed} cleaned entries.")

    prefiltered = pre_clean(results)
    print(f"🧹 Pre-cleaning reduced {len(results)} → {len(prefiltered)} potential names.")

    total = len(prefiltered)
    for i in tqdm(range(processed, total, BATCH_SIZE), desc="LLM name cleaning"):
        batch = prefiltered[i:i + BATCH_SIZE]
        names = [a["author_name"] for a in batch]

        prompt = (
            "You are a multilingual name validator. "
            "Given the following OCR-extracted names, return only valid personal names. "
            "Ignore names of concepts, organizations, names consisting of initials only, random capitalized words, capitalized Spanish common nouns, non-sense, or fragments. "
            "Keep one-name authors ONLY if they are clearly established authors like: "
            "Homero, Platón, Dante, Virgilio, Ovidio, Sócrates, Horacio, Catulo, Juvenal, Safo, or similar. "
            "Everything else that is one word should be discarded. "
            "Do not include any one-word name unless it is the name of an established or classic author you are aware of. "
            "You must exclude any one-word item that looks like a common noun, adjective, or arbitrary capitalized word, even if it could be a surname. "
            "If you include any name of one word that is not the name of an established or classic author the response will be invalid. "
            "Return a valid JSON array of clean names like [\"Name1\", \"Name2\"].\n\n"
            "Do not include any explanations, rationales, or anything else but the names themselves in the JSON array. "
            "If no valid names, return [].\n\n"
            f"Names: {json.dumps(names, ensure_ascii=False)}"
        )

        def run_generation_with_retry(prompt, retries=1):
            """Helper to retry model generation if JSON parsing fails."""
            try:
                response = generator(prompt, max_new_tokens=150, temperature=0.4)[0]["generated_text"]
                matches = re.findall(r"\[[^\]]*\]", response)
                if matches:
                    return json.loads(matches[-1])  # last JSON array
            except Exception as e:
                print(f"⚠️ JSON error, will retry: {e}")

            # 🔁 Retry once with safer parameters
            if retries > 0:
                try:
                    print("🔁 Retrying batch with safer decoding...")
                    response = generator(prompt, max_new_tokens=120, temperature=0.2)[0]["generated_text"]
                    matches = re.findall(r"\[[^\]]*\]", response)
                    if matches:
                        return json.loads(matches[-1])
                except Exception as e:
                    print(f"⚠️ Second attempt failed: {e}")
            return []

        valid_names = run_generation_with_retry(prompt)

        # 🧩 Apply fuzzy post-filter here
        before_filter = len(valid_names)
        valid_names = filter_one_word_names(valid_names)
        after_filter = len(valid_names)
        if before_filter != after_filter:
            print(f"🔍 Post-filter removed {before_filter - after_filter} spurious single-word names.")

        cleaned_batch = [
            {
                "author_name": n.strip(),
                "journal_index": batch[0].get("journal_index", "not_identified"),
                "journal_name": batch[0].get("journal_name", "not_identified"),
                "issue": batch[0].get("issue", "not_identified"),
                "date": batch[0].get("date", "not_identified"),
            }
            for n in valid_names if n.strip()
        ]
        cleaned.extend(cleaned_batch)

        # 💾 Autosave checkpoint
        if len(cleaned) % AUTOSAVE_EVERY < BATCH_SIZE:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(cleaned, f, ensure_ascii=False, indent=2)
            if BACKUP_PICKLE:
                with open(output_file.replace(".json", ".pkl"), "wb") as pf:
                    pickle.dump(cleaned, pf)
            print(f"💾 Autosaved after {len(cleaned)} names.")
        time.sleep(SLEEP_BETWEEN_BATCHES)

    # ✅ Final save
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    if BACKUP_PICKLE:
        with open(output_file.replace(".json", ".pkl"), "wb") as pf:
            pickle.dump(cleaned, pf)

    print(f"✅ Done! Cleaned {len(cleaned)} names saved to {output_file}")
    return cleaned


In [1]:
import pickle

with open("ahira_cmola_contributors_from_possible_tables_of_contents.pkl", 'rb') as file:
    results = pickle.load(file)

In [2]:
len(results)

535986

In [45]:
import re
import sys
sys.modules.pop('tqdm', None)
from tqdm import tqdm
import json
import time

In [57]:

cleaned_results = clean_author_names_with_checkpoint(results[:41])

🧹 Pre-cleaning reduced 41 → 30 potential names.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.?, ?it/s]


🔍 Post-filter removed 5 spurious single-word names.
💾 Autosaved after 5 names.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.7.69s/it]


🔍 Post-filter removed 6 spurious single-word names.
💾 Autosaved after 9 names.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.7.61s/it]


🔍 Post-filter removed 7 spurious single-word names.


LLM name cleaning: 100%|██████████████████████████| 3/3 [00:22<00:00,  7.61s/it]

✅ Done! Cleaned 12 names saved to ahira_cmola_cleaned_toc_authors_by_distilgpt2.json


In [58]:

cleaned_results

[{'author_name': 'Hstadós Uni',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Pos Ve',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Paseo Colón',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Hilario Virginio',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Joaquín Gras',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Pad Ea',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Juan de Garay',
  'journal_index': '980',
  'journ

In [ ]:

cleaned_results = clean_author_names_with_checkpoint(results)

In [ ]:

DONE CLEANING w LLM

In [9]:

import pickle
import json


with open("ahira_cmola_cleaned_toc_authors_by_distilgpt2.json", "r") as file:
    cleaned_results = json.load(file)

len(cleaned_results)

147284

In [ ]:

# THAT IS THE NUMBER OF CLEANED POTENTIAL CONTRIBUTORS EXTRACTED FROM POSSIBLE TABLES OF CONTENTS (OUT OF 535986)
# 147284 (OUT OF 535986)


In [2]:
cleaned_results[:20]

[{'author_name': 'Hstadós Uni',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Pos Ve',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Paseo Colón',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Hilario Virginio',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Joaquín Gras',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Pad Ea',
  'journal_index': '980',
  'journal_name': 'not_identified_980',
  'issue': 'not_identified',
  'date': 'not_identified'},
 {'author_name': 'Juan de Garay',
  'journal_index': '980',
  'journ

In [3]:

cleaned_results[1001]

{'author_name': 'Curlos Sánchez Viamonte',
 'journal_index': '172',
 'journal_name': 'La-Literatura-Argentina-12',
 'issue': 'La-Literatura-Argentina-12',
 'date': 'Agosto de 1929'}

In [ ]:

# # FURTHER CLEANING WITH ANOTHER LLM & rapidfuzz 


In [10]:

# List of allowed "connector" words (particles)
ALLOWED_PARTICLES = {
    # Spanish
    "y", "e", "de", "del", "la", "los", "las", "el",  

    # French
    "du", "de", "des", "d'",  

    # Italian  
    "di", "della", "del", "dei", "da", "d'",  

    # German & Dutch  
    "von", "van", "der", "den", "zum", "zur",  

    # English  
    "of", "the",  

    # Portuguese  
    "do", "dos", "da", "das",  

    # Arabic-origin names  
    "bin", "ibn", "al", "el",  

    # Others (Nordic, Celtic, Slavic, etc.)  
    "af", "ter", "zu", "mac", "mc"
}

In [11]:

from rapidfuzz import fuzz
from collections import defaultdict
import re



# ------------------------------------------------
# 1. Canonicalize Names (handles initials & memory)
# ------------------------------------------------

canonical_map = {}
history_map = {}

def canonicalize_name(cleaned_name, output_list, current_index):
    """Handle initials vs. full names retroactively."""
    if not cleaned_name:
        return {"author_name": None, "probably": False}

    tokens = cleaned_name.split()
    if len(tokens) < 2:
        return {"author_name": cleaned_name, "probably": True}

    # Detect compound last names
    if len(tokens) >= 2 and tokens[-2].lower() in ALLOWED_PARTICLES:
        last_name = tokens[-2] + " " + tokens[-1]
        first_tokens = tokens[:-2]
    else:
        last_name = tokens[-1]
        first_tokens = tokens[:-1]

    # Already seen last name?
    if last_name in canonical_map:
        canonical_full = canonical_map[last_name]
        if all(len(t) == 1 or t.endswith(".") for t in first_tokens):
            # initials only → adopt known canonical
            return {"author_name": canonical_full, "probably": True}
        else:
            # full name → retroactively fix initials
            canonical_map[last_name] = cleaned_name
            for idx in history_map.get(last_name, []):
                output_list[idx]["author_name"] = cleaned_name
                output_list[idx]["probably"] = True
            history_map[last_name] = []
            return {"author_name": cleaned_name, "probably": False}
    else:
        # new last name
        if all(len(t) == 1 or t.endswith(".") for t in first_tokens):
            partial = f"{first_tokens[0][0]}. {last_name}" if first_tokens else last_name
            history_map.setdefault(last_name, []).append(current_index)
            canonical_map[last_name] = partial
            return {"author_name": partial, "probably": True}
        else:
            canonical_map[last_name] = cleaned_name
            return {"author_name": cleaned_name, "probably": False}


In [19]:

import json
import pickle
from rapidfuzz import fuzz
from itertools import permutations
import os

def collapse_clusters_with_resume(cleaned_results, checkpoint_prefix="ahira_cmola_toc_collapsed_by_phi-2_and_rapidfuzz",
                                  threshold=80, checkpoint_every=50):
    """
    Collapse entries in cleaned_results based on similar author names.

    cleaned_results: list of dicts with 'author_name' and other metadata.
    checkpoint_prefix: filename prefix for periodic saving.
    threshold: similarity score threshold for clustering (0-100).
    checkpoint_every: save progress every N processed clusters.
    """
    # 1. Prepare name list and indices
    names = [r["author_name"] for r in cleaned_results if r["author_name"]]
    indices = [i for i, r in enumerate(cleaned_results) if r["author_name"]]

    # 2. Build clusters
    clusters = []
    visited = set()
    for i, name in enumerate(names):
        if indices[i] in visited:
            continue
        cluster = [indices[i]]
        visited.add(indices[i])
        for j in range(i + 1, len(names)):
            if indices[j] in visited:
                continue
            score = fuzz.ratio(name.lower(), names[j].lower())
            if score >= threshold:
                cluster.append(indices[j])
                visited.add(indices[j])
        if len(cluster) > 1:
            clusters.append(cluster)

    # 3. Process clusters and merge entries
    merged_results = []
    processed_clusters = 0

    for cluster_indices in clusters:
        # Suggest canonical name
        candidate_names = [cleaned_results[i]["author_name"] for i in cluster_indices]
        candidate_names.sort(key=lambda x: len(x), reverse=True)
        canonical_name = candidate_names[0]  # replace with model suggestion if desired

        merged_entry = {"author_name": canonical_name,
                        "initial_names": [],
                        "journal_index": [],
                        "journal_name": [],
                        "issue": [],
                        "date": []}

        for i in cluster_indices:
            entry = cleaned_results[i]
            merged_entry["initial_names"].append(entry["author_name"])
            for key in ["journal_index", "journal_name", "issue", "date"]:
                val = entry.get(key)
                if val is not None:
                    merged_entry[key].append(val)

        merged_results.append(merged_entry)
        processed_clusters += 1

        # Checkpoint periodically
        if processed_clusters % checkpoint_every == 0:
            json_file = f"{checkpoint_prefix}_checkpoint.json"
            pkl_file = f"{checkpoint_prefix}_checkpoint.pkl"
            with open(json_file, "w", encoding="utf-8") as f_json:
                json.dump(merged_results, f_json, ensure_ascii=False, indent=2)
            with open(pkl_file, "wb") as f_pkl:
                pickle.dump(merged_results, f_pkl)
            print(f"Checkpoint saved after {processed_clusters} clusters to {json_file} and {pkl_file}")

    # 4. Include entries not in any cluster
    all_clustered_indices = set(idx for c in clusters for idx in c)
    for i, entry in enumerate(cleaned_results):
        if i not in all_clustered_indices:
            single_entry = {"author_name": entry["author_name"],
                            "initial_names": [entry["author_name"]],
                            "journal_index": [entry.get("journal_index")],
                            "journal_name": [entry.get("journal_name")],
                            "issue": [entry.get("issue")],
                            "date": [entry.get("date")]}
            merged_results.append(single_entry)

    # Final save
    json_file = f"{checkpoint_prefix}_final.json"
    pkl_file = f"{checkpoint_prefix}_final.pkl"
    with open(json_file, "w", encoding="utf-8") as f_json:
        json.dump(merged_results, f_json, ensure_ascii=False, indent=2)
    with open(pkl_file, "wb") as f_pkl:
        pickle.dump(merged_results, f_pkl)
    print(f"Final merged output saved to {json_file} and {pkl_file}")

    return merged_results


In [13]:
# del model

In [12]:
!pip install -U bitsandbytes


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [8]:

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Use smaller model that fits in memory
model_name = "microsoft/phi-2"  # or "mistralai/Mistral-7B-Instruct-v0.2"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"  # Will use MPS on macOS
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [14]:

def suggest_canonical_from_cluster(cluster_indices, cleaned_results):
    """
    Suggest a canonical full name for a cluster of likely OCR/variant names.
    Uses Mistral-8x22B Instruct model with MPS + 4-bit quantization.
    """
    candidate_names = [cleaned_results[i]["author_name"] for i in cluster_indices]
    candidate_names.sort(key=lambda x: len(x), reverse=True)
    initial_candidate = candidate_names[0]

    prompt = (
        "These names are likely OCR or typographical variants of the same person:\n"
        f"{candidate_names}\n\n"
        "Suggest the most probable correct full name. "
        "Keep correct capitalization and accents. Return ONLY the corrected name."
    )

    try:
        # Generate corrected name
        out = pipe(prompt, max_new_tokens=50, temperature=0.2)[0]["generated_text"]
        corrected_name = out.strip().split("\n")[0]
        if not corrected_name:
            corrected_name = initial_candidate
    except Exception as e:
        print(f"Error in model inference: {e}")
        corrected_name = initial_candidate

    # Update all entries in cluster
    for i in cluster_indices:
        cleaned_results[i]["author_name"] = corrected_name
        cleaned_results[i]["probably"] = True

    return corrected_name


In [17]:

i = 0

for idx, entry in enumerate(cleaned_results):
    res = canonicalize_name(entry["author_name"], cleaned_results, idx)
    if entry["author_name"] != res["author_name"]:
        i += 1
    entry["author_name"] = res["author_name"]
    entry["probably"] = res.get("probably", False)


In [18]:
i

241

In [ ]:

collapsed_results = collapse_clusters_with_resume(cleaned_results)

In [21]:

len(collapsed_results)

63544

In [ ]:

# SO, FURTHER WHITTLED DOWN (COLLAPSED) TO 63544 !!!!!!!!!!!!!!!!!!!


In [ ]:

# WIKIDATA SEARCHES


In [2]:

import json
import requests
import time
from itertools import permutations
import difflib
import rapidfuzz

wikidata_cache = {}

In [5]:
 
with open("ahira_cmola_toc_collapsed_by_phi-2_and_rapidfuzz_final.json", "r", encoding="utf-8") as f:
    contributors_cleaned_by_phi2 = json.load(f)

f.close()

In [11]:

DOMAIN_KEYWORDS = {
    # Literature / writing / publishing
        "writer", "author", "journalist", "poet", "novelist", "essayist", "playwright", 
        "translator", "editor", "literary critic", "critic", "biographer", "columnist", "reporter",
        "travel writer", "diarist",

    # Visual arts
        "artist", "painter", "sculptor", "illustrator", "engraver", "photographer", 
        "cartoonist", "calligrapher", "designer", "ceramist",

    # Performing arts
        "filmmaker", "director", "screenwriter", "actor", "actress", "play director", 
        "theatre director", "dancer", "choreographer", "musician", "composer", "singer", 
        "conductor", "film director",

    # Academic / intellectual / science
        "scientist", "physicist", "chemist", "biologist", "anthropologist", "historian", 
        "philosopher", "psychologist", "psychiatrist", "psychoanalyst", "sociologist", 
        "linguist", "economist", "mathematician", "archaeologist", "researcher", "professor", "teacher",

    # Architecture / design
        "architect", "urbanist", "engineer", "cartographer", "designer",

    # Social / political / cultural activism
        "politician", "activist", "feminist", "revolutionary", "journal editor", 
        "humanist", "cultural promoter",

    # Popular science / communication
        "science communicator", "naturalist", "astronomer", "popularizer", "explorer",

    # Misc. related roles
        "collector", "curator", "librarian", "museum director", "art critic", "film critic",
        "photography critic", "art historian", "ethnographer"

}

In [19]:
from collections import defaultdict
from typing import List, Dict, Optional

In [14]:

from rapidfuzz import fuzz

def similarity_score(a: str, b: str) -> float:
    """
    Compute a token-based similarity score between two labels using RapidFuzz.

    """
    if not a or not b:
        return 0.0

    return fuzz.token_set_ratio(a, b) / 100.0

def query_wikidata_cached(name, domain_hint=None):
    """
   Cached Wikidata search with scoring, domain boosting, fuzzy fallback.
    Returns (best_hit_dict or None, confidence_score)
    """
    if name in wikidata_cache:
        return wikidata_cache[name]

    base_url = "https://www.wikidata.org/w/api.php"
    session = requests.Session()
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "limit": 5,
        "search": name
    }

    HEADERS = {
    "User-Agent": "WikidataAuthorQuery/1.0 (ctanasescu@uoc.edu; GLobal Literary Studies Research Group, Universitat Oberta de Catalunya) Python requests"
    }

    results = []
    try:
        r = session.get(base_url, params=params, headers=HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()
        results.extend(data.get("search", []))
    except (requests.RequestException, ValueError) as e:
        print(f"⚠️ Error querying '{name}': {e}")
        wikidata_cache[name] = (None, 0)
        return (None, 0)

    # --- scoring & domain boosting ---
    best_match, best_conf = None, 0.0
    for r in results:
        label = r.get("label", "")
        desc = (r.get("description") or "").lower()
        score = similarity_score(label, name)
        if any(word in desc for word in DOMAIN_KEYWORDS):
            score += 0.1
        score = min(score, 1.0)
        if score > best_conf:
            best_conf = score
            best_match = r

    # --- fuzzy fallback if needed ---
    if best_conf < 0.8 and len(name.split()) > 1:
        tokens = name.split()
        variants = set()
        variants.add(" ".join(tokens[::-1]))
        for perm in permutations(tokens, min(2, len(tokens))):
            variants.add(" ".join(perm))

        for variant in variants:
            params["search"] = variant
            try:
                r = session.get(base_url, params=params, headers=HEADERS, timeout=10)
                r.raise_for_status()
                data = r.json()
                for res in data.get("search", []):
                    label = res.get("label", "")
                    desc = (res.get("description") or "").lower()
                    score = similarity_score(label, name)
                    if any(word in desc for word in DOMAIN_KEYWORDS):
                        score += 0.1
                    score = min(score, 1.0)
                    if score > best_conf:
                        best_conf = score
                        best_match = res
            except:
                continue

    if best_conf < 0.5 and best_match:
        best_conf = 0.5

    # --- cache and return ---
    wikidata_cache[name] = (best_match, best_conf)
    time.sleep(0.3)  # avoid hammering API
    return best_match, best_conf


def wikidata_search_label_safe(name, lang="*", limit=5):
    """
    Safe wrapper around existing Wikidata search.
    Returns list of hits or empty list if the request fails.
    """
    base_url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": lang,
        "format": "json",
        "limit": limit
    }
    try:
        r = requests.get(base_url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        return data.get("search", [])
    except (requests.RequestException, ValueError) as e:
        print(f"⚠️ Error querying '{name}': {e}")
        return []


In [25]:


def wikidata_author_metadata(qid: str) -> Dict:
    """
    Fetch enriched author metadata from Wikidata for a single author QID.
    Returns a dict with:
    - is_human (bool)
    - labels (list of labels from Wikidata)
    - gender
    - date_of_birth
    - date_of_death
    - occupations (list)
    - countries (list)
    - languages_written (list)
    - genres (list)
    - prevailing_genre (genre with most works)
    """

    # Safety check
    if not qid or not qid.startswith("Q"):
        return {
            'is_human': False,
            'labels': [],
            'gender': None,
            'dob': None,
            'dod': None,
            'occupations': [],
            'countries': [],
            'languages_written': [],
            'genres': [],
            'prevailing_genre': None
        }

    HEADERS = {
        "User-Agent": "WikidataAuthorQuery/1.0 (ctanasescu@uoc.edu; GLobal Literary Studies Research Group, Universitat Oberta de Catalunya) Python requests"
    }

    # ✅ SPARQL query with human check and safe ORDER BY
    sparql = f"""
    SELECT * WHERE {{
      {{
        SELECT ?author ?authorLabel ?genderLabel ?dob ?dod ?countryLabel ?occupationLabel ?languageLabel ?genreLabel ?instanceLabel (COUNT(?work) AS ?genreCount)
        WHERE {{
            VALUES ?author {{ wd:{qid} }}
            OPTIONAL {{ ?author wdt:P31 ?instance. }}
            OPTIONAL {{ ?author wdt:P21 ?gender. }}
            OPTIONAL {{ ?author wdt:P569 ?dob. }}
            OPTIONAL {{ ?author wdt:P570 ?dod. }}
            OPTIONAL {{ ?author wdt:P106 ?occupation. }}
            OPTIONAL {{ ?author wdt:P27 ?country. }}
            OPTIONAL {{ ?author wdt:P407 ?language. }}
            OPTIONAL {{
                ?work wdt:P50 ?author.
                ?work wdt:P136 ?genre.
            }}
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,es,fr,de,pt,ro,it,sv,no,da,fi,pl,uk,ru". }}
        }}
        GROUP BY ?author ?authorLabel ?genderLabel ?dob ?dod ?countryLabel ?occupationLabel ?languageLabel ?genreLabel ?instanceLabel
      }}
    }}
    ORDER BY DESC(?genreCount)
    """

    url = "https://query.wikidata.org/sparql"

    time.sleep(0.3)  # 300 ms between requests

    try:
        r = requests.get(url, params={'query': sparql, 'format': 'json'}, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
    except requests.HTTPError as e:
        print(f"⚠️ SPARQL query failed for {qid}: {e}")
        return _empty_author_dict()
    except requests.RequestException as e:
        print(f"⚠️ Network error for {qid}: {e}")
        return _empty_author_dict()

    bindings = data.get('results', {}).get('bindings', [])
    if not bindings:
        return _empty_author_dict()

    # --- Aggregate results ---
    result = {
        'is_human': False,
        'labels': [],
        'gender': None,
        'dob': None,
        'dod': None,
        'occupations': [],
        'countries': [],
        'languages_written': [],
        'genres': [],
        'prevailing_genre': None
    }

    max_genre_count = -1
    for b in bindings:
        # Labels
        if 'authorLabel' in b:
            label_val = b['authorLabel']['value']
            if label_val not in result['labels']:
                result['labels'].append(label_val)
        # Instance of (to check if human)
        if 'instanceLabel' in b:
            inst = b['instanceLabel']['value'].lower()
            if "human" in inst or "humain" in inst:
                result['is_human'] = True
        # Gender
        if 'genderLabel' in b:
            result['gender'] = b['genderLabel']['value']
        # Birth/Death
        if 'dob' in b:
            result['dob'] = b['dob']['value']
        if 'dod' in b:
            result['dod'] = b['dod']['value']
        # Occupations
        if 'occupationLabel' in b:
            occ = b['occupationLabel']['value']
            if occ not in result['occupations']:
                result['occupations'].append(occ)
        # Countries
        if 'countryLabel' in b:
            c = b['countryLabel']['value']
            if c not in result['countries']:
                result['countries'].append(c)
        # Languages
        if 'languageLabel' in b:
            lang = b['languageLabel']['value']
            if lang not in result['languages_written']:
                result['languages_written'].append(lang)
        # Genres
        if 'genreLabel' in b:
            genre = b['genreLabel']['value']
            if genre not in result['genres']:
                result['genres'].append(genre)
            # Prevailing genre
            count = int(b.get('genreCount', {}).get('value', 0))
            if count > max_genre_count:
                max_genre_count = count
                result['prevailing_genre'] = genre

    return result


def _empty_author_dict():
    """Helper for empty or failed metadata lookup."""
    return {
        'is_human': False,
        'labels': [],
        'gender': None,
        'dob': None,
        'dod': None,
        'occupations': [],
        'countries': [],
        'languages_written': [],
        'genres': [],
        'prevailing_genre': None
    }


In [ ]:

# (COUNT(?work) AS ?genreCount)
# ORDER BY DESC(?genreCount)

In [43]:

import random

MAX_RETRIES = 5

METADATA_CACHE_FILE = "ahira_cmola_wikidata_metadata_cache.json"

if os.path.exists(METADATA_CACHE_FILE):
    with open(METADATA_CACHE_FILE, "r", encoding="utf-8") as f:
        wikidata_metadata_cache = json.load(f)
else:
    wikidata_metadata_cache = {}

def save_metadata_cache():
    with open(METADATA_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(wikidata_metadata_cache, f, ensure_ascii=False, indent=2)

def _empty_author_dict():
    return {
        "is_human": False,
        "gender": None,
        "country": None,
        "occupation": None,
        "language": None,
        "genre": None,
        "dob": None,
        "dod": None,
        # "prevailing_genre": None,
    }

def wikidata_author_metadata(qid: str) -> Dict:
    if not qid or not qid.startswith("Q"):
        return {
            'is_human': False,
            'labels': [],
            'gender': None,
            'dob': None,
            'dod': None,
            'occupations': [],
            'countries': [],
            'languages_written': [],
            'genres': [],
            # 'prevailing_genre': None
        }

    HEADERS = {
        "User-Agent": "WikidataAuthorQuery/1.0 (ctanasescu@uoc.edu; GLobal Literary Studies Research Group, Universitat Oberta de Catalunya) Python requests"
    }

    if qid in wikidata_metadata_cache:
        return wikidata_metadata_cache[qid]
        
    sparql = f"""
    SELECT * WHERE {{
      {{
        SELECT ?author ?authorLabel ?genderLabel ?dob ?dod ?countryLabel ?occupationLabel ?languageLabel ?genreLabel ?instanceLabel 
        WHERE {{
            VALUES ?author {{ wd:{qid} }}
            OPTIONAL {{ ?author wdt:P31 ?instance. }}
            OPTIONAL {{ ?author wdt:P21 ?gender. }}
            OPTIONAL {{ ?author wdt:P569 ?dob. }}
            OPTIONAL {{ ?author wdt:P570 ?dod. }}
            OPTIONAL {{ ?author wdt:P106 ?occupation. }}
            OPTIONAL {{ ?author wdt:P27 ?country. }}
            OPTIONAL {{ ?author wdt:P407 ?language. }}
            OPTIONAL {{
                ?work wdt:P50 ?author.
                ?work wdt:P136 ?genre.
            }}
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,es,fr,de,pt,ro,it,sv,no,da,fi,pl,uk,ru". }}
        }}
        GROUP BY ?author ?authorLabel ?genderLabel ?dob ?dod ?countryLabel ?occupationLabel ?languageLabel ?genreLabel ?instanceLabel
      }}
    }}
    """
    url = "https://query.wikidata.org/sparql"

    delay = 1.0
    MAX_RETRIES = 5
    data = None

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(
                url,
                params={"query": sparql, "format": "json"},
                headers=HEADERS,
                timeout=60
            )
            if r.status_code == 429:
                print(f"⏳ Rate limited on attempt {attempt+1} for {qid}. Sleeping {delay:.1f}s.")
                time.sleep(delay + random.uniform(0, 0.5))
                delay *= 2
                continue
            r.raise_for_status()
            data = r.json()
            break
        except requests.ReadTimeout:
            print(f"⚠️ Timeout on {qid}, retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= 1.5
            continue
        except requests.RequestException as e:
            print(f"❌ Network error for {qid}: {e}")
            return _empty_author_dict()

    if data is None:
        # Après tous les essais, rien
        result = _empty_author_dict()
        wikidata_metadata_cache[qid] = result
        save_metadata_cache()
        return result

    results = data.get("results", {}).get("bindings", [])
    if not results:
        result = _empty_author_dict()
        wikidata_metadata_cache[qid] = result
        save_metadata_cache()
        return result

    row = results[0]
    is_human = row.get("authorLabel", {}).get("value") is not None

    result = {
        "is_human": is_human,
        "gender": row.get("genderLabel", {}).get("value"),
        "country": row.get("countryLabel", {}).get("value"),
        "occupation": row.get("occupationLabel", {}).get("value"),
        "language": row.get("languageLabel", {}).get("value"),
        "genre": row.get("genreLabel", {}).get("value"),
        "dob": row.get("dob", {}).get("value"),
        "dod": row.get("dod", {}).get("value"),
        # "prevailing_genre": row.get("genreLabel", {}).get("value"),
    }

    # Ajouter au cache
    wikidata_metadata_cache[qid] = result
    save_metadata_cache()

    return result

In [46]:

def wikidata_author_metadata(qid: str) -> Dict:
    if not qid or not qid.startswith("Q"):
        return _empty_author_dict()

    if qid in wikidata_metadata_cache:
        return wikidata_metadata_cache[qid]

    HEADERS = {
        "User-Agent": "WikidataAuthorQuery/1.0 (ctanasescu@uoc.edu; GLobal Literary Studies Research Group, Universitat Oberta de Catalunya) Python requests"
    }

    sparql = f"""
    SELECT ?author ?authorLabel ?genderLabel ?dob ?dod ?countryLabel ?occupationLabel ?languageLabel ?genreLabel ?instanceLabel WHERE {{
        VALUES ?author {{ wd:{qid} }}
        OPTIONAL {{ ?author wdt:P31 ?instance. }}
        OPTIONAL {{ ?author wdt:P21 ?gender. }}
        OPTIONAL {{ ?author wdt:P569 ?dob. }}
        OPTIONAL {{ ?author wdt:P570 ?dod. }}
        OPTIONAL {{ ?author wdt:P106 ?occupation. }}
        OPTIONAL {{ ?author wdt:P27 ?country. }}
        OPTIONAL {{ ?author wdt:P407 ?language. }}
        OPTIONAL {{
            ?work wdt:P50 ?author.
            ?work wdt:P136 ?genre.
        }}
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,es,fr,de,pt,ro,it,sv,no,da,fi,pl,uk,ru". }}
    }}
    """

    url = "https://query.wikidata.org/sparql"
    delay = 1.0
    MAX_RETRIES = 5
    data = None

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params={"query": sparql, "format": "json"},
                             headers=HEADERS, timeout=60)
            if r.status_code == 429:
                print(f"⏳ Rate limited on attempt {attempt+1} for {qid}. Sleeping {delay:.1f}s.")
                time.sleep(delay + random.uniform(0, 0.5))
                delay *= 2
                continue
            r.raise_for_status()
            data = r.json()
            break
        except requests.ReadTimeout:
            print(f"⚠️ Timeout on {qid}, retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= 1.5
        except requests.RequestException as e:
            print(f"❌ Network error for {qid}: {e}")
            return _empty_author_dict()

    if data is None:
        result = _empty_author_dict()
        wikidata_metadata_cache[qid] = result
        save_metadata_cache()
        return result

    bindings = data.get("results", {}).get("bindings", [])
    if not bindings:
        result = _empty_author_dict()
        wikidata_metadata_cache[qid] = result
        save_metadata_cache()
        return result

    # --- Aggregate results manually ---
    result = _empty_author_dict()
    result["is_human"] = False
    labels, occs, langs, genres, countries = set(), set(), set(), set(), set()
    gender = dob = dod = None

    for b in bindings:
        if "authorLabel" in b:
            labels.add(b["authorLabel"]["value"])
        if "genderLabel" in b:
            gender = b["genderLabel"]["value"]
        if "dob" in b:
            dob = b["dob"]["value"]
        if "dod" in b:
            dod = b["dod"]["value"]
        if "occupationLabel" in b:
            occs.add(b["occupationLabel"]["value"])
        if "languageLabel" in b:
            langs.add(b["languageLabel"]["value"])
        if "countryLabel" in b:
            countries.add(b["countryLabel"]["value"])
        if "genreLabel" in b:
            genres.add(b["genreLabel"]["value"])
        if "instanceLabel" in b and "human" in b["instanceLabel"]["value"].lower():
            result["is_human"] = True

    result.update({
        "is_human": result["is_human"] or True,  # assume human if reachable
        "labels": list(labels),
        "gender": gender,
        "dob": dob,
        "dod": dod,
        "occupations": list(occs),
        "countries": list(countries),
        "languages_written": list(langs),
        "genres": list(genres),
    })

    wikidata_metadata_cache[qid] = result
    save_metadata_cache()
    return result


In [8]:

contributors_cleaned_by_phi2[100]

{'author_name': 'Otto Miguel',
 'initial_names': ['Otto Miguel',
  'Otto Miguel',
  'Otto Miguet',
  'Otto Miguel',
  'Otto Miguel',
  'Otto Miguel',
  'Otto Miguel'],
 'journal_index': [3313, '4200', 2406, 2535, 458, 3281, '2640'],
 'journal_name': ['Fray mocho',
  'not_identified_4200',
  'Mundo argentino',
  'Mundo argentino',
  'PBT',
  'Fray mocho',
  'not_identified_2640'],
 'issue': ['9.1920,3.Febr.=Nr. 406',
  'not_identified',
  '4.1914,28.Jan.=Nr. 160',
  '12.1922,2.Aug.=Nr. 602',
  'Indice 12.1915,6.Mrz./24.Apr.',
  '19.1931,Jan.=Nr. 922',
  'not_identified'],
 'date': ['9.1920,3.Febr.=Nr. 406',
  'not_identified',
  '4.1914,28.Jan.=Nr. 160',
  '12.1922,2.Aug.=Nr. 602',
  'Indice 12.1915,6.Mrz./24.Apr.',
  '19.1931,Jan.=Nr. 922',
  'not_identified']}

In [27]:

import pickle
import os

def save_progress(file_prefix, resolved, not_found, missing_fields, non_spanish, multiple_candidates):
    with open(f"{file_prefix}_resolved.pkl", "wb") as f:
        pickle.dump(resolved, f)
    with open(f"{file_prefix}_not_found.pkl", "wb") as f:
        pickle.dump(not_found, f)
    with open(f"{file_prefix}_missing_fields.pkl", "wb") as f:
        pickle.dump(missing_fields, f)
    with open(f"{file_prefix}_non_spanish.pkl", "wb") as f:
        pickle.dump(non_spanish, f)
    with open(f"{file_prefix}_multiple_candidates.pkl", "wb") as f:
        pickle.dump(multiple_candidates, f)
    print(f"Progress saved to {file_prefix}_*.pkl")

In [28]:

resolved_list = []
not_found_list = []
missing_fields_list = []
non_spanish_list = []
multiple_candidates_list = []

In [ ]:

CACHE_FILE = "ahira_cmola_toc_wikidata_cache.pkl"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        wikidata_cache = pickle.load(f)
else:
    wikidata_cache = {}

# Helper to persist it
def save_cache():
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(wikidata_cache, f)

for i, candidate in enumerate(contributors_cleaned_by_phi2[2:]):
    cleaned_name = candidate.get("author_name")

    if not cleaned_name or str(cleaned_name).lower() == "null":
        not_found_list.append(candidate)
        continue

    # Try Wikidata search with main author_name
    wd_hit, confidence = query_wikidata_cached(cleaned_name)

    # Fallback: try unique names in 'initial_names' if no result
    if not wd_hit:
        initial_names = candidate.get("initial_names", [])
        if isinstance(initial_names, list):
            # Use a set to avoid duplicates, and preserve order if possible
            tried = set()
            for alt_name in initial_names:
                alt_name = alt_name.strip()
                if not alt_name or alt_name.lower() in tried:
                    continue
                tried.add(alt_name.lower())

                wd_hit, confidence = query_wikidata_cached(alt_name)
                if wd_hit:
                    # Overwrite cleaned_name with successful alternative
                    cleaned_name = alt_name
                    break

    # If still no match, mark as not found
    if not wd_hit:
        not_found_list.append(candidate)
        continue

    qid = wd_hit["id"]
    label = wd_hit.get("label")
    description = wd_hit.get("description")

    meta = wikidata_author_metadata(qid)
    
    if not meta["is_human"]:
        not_found_list.append(candidate)
        continue

    chosen_qid, chosen_meta = qid, meta

    missing_fields = {
        "gender": chosen_meta["gender"] is None,
        "languages_written": not bool(chosen_meta["languages_written"]),
        "genres": not bool(chosen_meta["genres"])
    }
    if any(missing_fields.values()):
        missing_fields_list.append({"candidate": candidate, "missing": missing_fields})

    if chosen_meta["languages_written"]:
        languages_lower = [l.lower() for l in chosen_meta["languages_written"]]
        if not any(x in languages_lower for x in ["spanish", "español", "castellano"]):
            non_spanish_list.append({"candidate": candidate, "languages": chosen_meta["languages_written"]})

    resolved_list.append({
        **candidate,
        **chosen_meta,
        "wikidata_id": qid,
        "wikidata_label": label,
        "wikidata_description": description,
        "match_confidence": round(confidence, 3),
        "matched_name": cleaned_name,  # optional: store which name succeeded
    })

    time.sleep(2.0)
    
    if i % 50 == 0:
        save_progress(
            "ahira_cmola_toc_contributors_by_wiki_via_distilgpt2_n_phi2",
            resolved_list, not_found_list, missing_fields_list, non_spanish_list, multiple_candidates_list
        )
        save_cache()
        print(i)


In [47]:

for i, candidate in enumerate(contributors_cleaned_by_phi2[2:]):
    cleaned_name = candidate.get("author_name")
    if not cleaned_name or str(cleaned_name).lower() == "null":
        not_found_list.append(candidate)
        continue

    # --- Rechercher sur Wikidata (cache + scoring) ---
    wd_hit, confidence = query_wikidata_cached(cleaned_name)

    # Fallback avec initial_names si pas trouvé
    if not wd_hit:
        initial_names = candidate.get("initial_names", [])
        if isinstance(initial_names, list):
            tried = set()
            for alt_name in initial_names:
                alt_name = alt_name.strip()
                if not alt_name or alt_name.lower() in tried:
                    continue
                tried.add(alt_name.lower())
                wd_hit, confidence = query_wikidata_cached(alt_name)
                if wd_hit:
                    cleaned_name = alt_name
                    break

    # --- Si pas de résultat, on marque comme non trouvé ---
    if not wd_hit:
        not_found_list.append(candidate)
        continue

    # --- Vérifier si la description contient un mot du domaine ---
    description = (wd_hit.get("description") or "").lower()
    if not any(word in description for word in DOMAIN_KEYWORDS):
        not_found_list.append(candidate)
        continue

    # --- Récupérer les métadonnées (peut être partielle si timeout) ---
    qid = wd_hit["id"]
    label = wd_hit.get("label")
    meta = wikidata_author_metadata(qid) or _empty_author_dict()

    # --- Corriger les faux négatifs "non-humains" ---
    if not meta.get("is_human", False):
        if any(word in (description or "") for word in DOMAIN_KEYWORDS):
            meta["is_human"] = True
            print(f"⚠️ Corrected non-human flag for {label} ({qid})")
    else:
        not_found_list.append(candidate)
        continue

    # --- Ajouter au resolved_list même si certains champs manquent ---
    resolved_entry = {
        **candidate,
        **meta,
        "wikidata_id": qid,
        "wikidata_label": label,
        "wikidata_description": wd_hit.get("description"),
        "match_confidence": round(confidence, 3),
        "matched_name": cleaned_name,
    }
    resolved_list.append(resolved_entry)

    # --- Tracker les champs manquants ---
    missing_fields = {
        "gender": meta.get("gender") is None,
        "languages_written": not bool(meta.get("languages_written") or meta.get("language")),
        "genres": not bool(meta.get("genres") or meta.get("genre"))
    }
    if any(missing_fields.values()):
        missing_fields_list.append({"candidate": candidate, "missing": missing_fields})

    # --- Identifier si langue principale non espagnole ---
    if meta.get("languages_written") or meta.get("language"):
        languages_lower = [l.lower() for l in (meta.get("languages_written") or [meta.get("language")])]
        if not any(x in languages_lower for x in ["spanish", "español", "castellano"]):
            non_spanish_list.append({"candidate": candidate, "languages": meta.get("languages_written") or [meta.get("language")]})

    # --- Sauvegarder le cache et progresser toutes les 50 itérations ---
    if i % 50 == 0 and i > 0:
        save_cache()
        print(f"Progress saved at index {i} ({len(resolved_list)} resolved, {len(not_found_list)} not found)")

    # --- Petite pause pour respecter la limite d'API ---
    time.sleep(1.0)

# --- Sauvegarde finale du cache ---
save_cache()
print("All done! Total resolved:", len(resolved_list))

⚠️ Error querying 'Eduardo Elnef Pretender': HTTPSConnectionPool(host='www.wikidata.org', port=443): Read timed out. (read timeout=10)
⚠️ Error querying 'Isidro Méndez': HTTPSConnectionPool(host='www.wikidata.org', port=443): Read timed out. (read timeout=10)
⚠️ Timeout on Q312829, retrying in 1.0s...
⚠️ Timeout on Q47389138, retrying in 1.0s...
⚠️ Timeout on Q461198, retrying in 1.0s...
⚠️ Timeout on Q4792778, retrying in 1.0s...
⚠️ Timeout on Q55458158, retrying in 1.0s...
⚠️ Error querying 'De Lasagna': HTTPSConnectionPool(host='www.wikidata.org', port=443): Read timed out. (read timeout=10)
⚠️ Timeout on Q64225589, retrying in 1.0s...
All done! Total resolved: 8


In [62]:
len(contributors_cleaned_by_phi2)

63544

In [56]:
len(not_found_list)

63597

In [57]:
len(non_spanish_list)

0

In [58]:
len(multiple_candidates_list)

0

In [59]:
len(missing_fields_list)

8

In [60]:
len(resolved_list)

8

In [ ]:
"ahira_cmola_toc_contributors_by_wiki_via_distilgpt2_n_phi2",
            resolved_list, not_found_list, missing_fields_list, non_spanish_list, multiple_candidates_list

In [53]:
resolved_list

[{'author_name': 'Joaquín Granel',
  'initial_names': ['Joaquín Gras',
   'Joaquín Reyes',
   'Joaquín Ramos',
   'Joaquín País',
   'Joaquín Granel',
   'Joaquín Gurefa',
   'Joaquín Uria',
   'Joaquín Casás',
   'Joaquín García',
   'Joaquín García',
   'Joaquín Arenas'],
  'journal_index': ['980',
   '467',
   '869',
   915,
   2363,
   1541,
   1089,
   3390,
   3184,
   '795',
   2593],
  'journal_name': ['not_identified_980',
   'Nervio-34-35',
   'El-40-5',
   'Céltiga',
   'PBT',
   'Céltiga',
   'Céltiga',
   'Céltiga',
   'Céltiga',
   'La-Literatura-Argentina-26',
   'Fray mocho'],
  'issue': ['not_identified',
   'Nervio-34-35',
   'El-40-5',
   '9.1932=Nr. 181/182',
   '2.1905,24.Jun.=Nr. 40',
   '4.1927=Nr. 57',
   '7.1930=Nr. 127',
   '5.1928=Nr. 82',
   '4.1927=Nr. 50',
   'La-Literatura-Argentina-26',
   '12.1923,27.Febr.=Nr. 566'],
  'date': ['not_identified',
   'Julio de 1934',
   'Verano-otoño de 1953',
   '9.1932=Nr. 181/182',
   '2.1905,24.Jun.=Nr. 40',
   '4.192

In [61]:

missing_fields_list

[{'candidate': {'author_name': 'Joaquín Granel',
   'initial_names': ['Joaquín Gras',
    'Joaquín Reyes',
    'Joaquín Ramos',
    'Joaquín País',
    'Joaquín Granel',
    'Joaquín Gurefa',
    'Joaquín Uria',
    'Joaquín Casás',
    'Joaquín García',
    'Joaquín García',
    'Joaquín Arenas'],
   'journal_index': ['980',
    '467',
    '869',
    915,
    2363,
    1541,
    1089,
    3390,
    3184,
    '795',
    2593],
   'journal_name': ['not_identified_980',
    'Nervio-34-35',
    'El-40-5',
    'Céltiga',
    'PBT',
    'Céltiga',
    'Céltiga',
    'Céltiga',
    'Céltiga',
    'La-Literatura-Argentina-26',
    'Fray mocho'],
   'issue': ['not_identified',
    'Nervio-34-35',
    'El-40-5',
    '9.1932=Nr. 181/182',
    '2.1905,24.Jun.=Nr. 40',
    '4.1927=Nr. 57',
    '7.1930=Nr. 127',
    '5.1928=Nr. 82',
    '4.1927=Nr. 50',
    'La-Literatura-Argentina-26',
    '12.1923,27.Febr.=Nr. 566'],
   'date': ['not_identified',
    'Julio de 1934',
    'Verano-otoño de 1953',
 

In [13]:
resolved_list = [{'author_name': 'Joaquín Granel',
  'initial_names': ['Joaquín Gras',
   'Joaquín Reyes',
   'Joaquín Ramos',
   'Joaquín País',
   'Joaquín Granel',
   'Joaquín Gurefa',
   'Joaquín Uria',
   'Joaquín Casás',
   'Joaquín García',
   'Joaquín García',
   'Joaquín Arenas'],
  'journal_index': ['980',
   '467',
   '869',
   915,
   2363,
   1541,
   1089,
   3390,
   3184,
   '795',
   2593],
  'journal_name': ['not_identified_980',
   'Nervio-34-35',
   'El-40-5',
   'Céltiga',
   'PBT',
   'Céltiga',
   'Céltiga',
   'Céltiga',
   'Céltiga',
   'La-Literatura-Argentina-26',
   'Fray mocho'],
  'issue': ['not_identified',
   'Nervio-34-35',
   'El-40-5',
   '9.1932=Nr. 181/182',
   '2.1905,24.Jun.=Nr. 40',
   '4.1927=Nr. 57',
   '7.1930=Nr. 127',
   '5.1928=Nr. 82',
   '4.1927=Nr. 50',
   'La-Literatura-Argentina-26',
   '12.1923,27.Febr.=Nr. 566'],
  'date': ['not_identified',
   'Julio de 1934',
   'Verano-otoño de 1953',
   '9.1932=Nr. 181/182',
   '2.1905,24.Jun.=Nr. 40',
   '4.1927=Nr. 57',
   '7.1930=Nr. 127',
   '5.1928=Nr. 82',
   '4.1927=Nr. 50',
   'octubre de 1930',
   '12.1923,27.Febr.=Nr. 566'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'prevailing_genre': None,
  'wikidata_id': 'Q18812825',
  'wikidata_label': 'Joaquín Granel',
  'wikidata_description': 'Argentinian politician',
  'match_confidence': 1.0,
  'matched_name': 'Joaquín Granel'},
 {'author_name': 'Carmen Lobo Arraga',
  'initial_names': ['Carmen Lobo Arraga', 'Carmen Lobo Arraga'],
  'journal_index': ['980', '980'],
  'journal_name': ['not_identified_980', 'not_identified_980'],
  'issue': ['not_identified', 'not_identified'],
  'date': ['not_identified', 'not_identified'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q83346379',
  'wikidata_label': 'Carmen Lobo Bedmar',
  'wikidata_description': 'researcher',
  'match_confidence': 0.859,
  'matched_name': 'Carmen Lobo Arraga'},
 {'author_name': 'Heliográficos de Ricardo',
  'initial_names': ['Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de Ri',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de Ricardo',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de',
   'Heliográficos de'],
  'journal_index': ['980',
   2687,
   3928,
   2687,
   3928,
   1948,
   1917,
   1071,
   '2659',
   2726,
   4488,
   '3009',
   3160,
   3897,
   2597,
   2322,
   4037,
   '165',
   '3321',
   711,
   2859,
   1634,
   83,
   2388,
   2406,
   2112,
   4288,
   713,
   2798,
   '3799',
   4484,
   4148,
   2378,
   3174,
   '3629',
   1259,
   4276,
   1734,
   2908,
   2549,
   1018,
   '790',
   120,
   3812,
   4573,
   4688,
   334,
   '2759',
   2398,
   2622,
   166,
   '480',
   1299,
   1626,
   1485,
   296,
   4154,
   2364,
   2749,
   1508,
   36,
   1039,
   4595,
   314,
   3230,
   3988,
   2531,
   3263,
   '2790',
   2958,
   1472,
   3759,
   3710,
   3344,
   890,
   4314,
   612,
   3006,
   56,
   '2621',
   2610,
   3874,
   2162,
   2993,
   '2640',
   4284,
   4639,
   2410,
   719,
   1826,
   3385],
  'journal_name': ['not_identified_980',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_2659',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_3009',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_165',
   'not_identified_3321',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_3799',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_3629',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_790',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_2759',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_480',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_2790',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_2621',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'not_identified_2640',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino',
   'Mundo argentino'],
  'issue': ['not_identified',
   '2.1912,11.Dez.=Nr. 101',
   '5.1915,7.Jul.=Nr. 235',
   '2.1912,11.Dez.=Nr. 101',
   '5.1915,7.Jul.=Nr. 235',
   '5.1915,31.Mrz.=Nr. 221',
   '5.1915,10.Mrz.=Nr. 218',
   '6.1916,16.Febr.=Nr. 267',
   'not_identified',
   '3.1913,25.Jun.=Nr. 129',
   '4.1914,23.Sept.=Nr. 194',
   'not_identified',
   '5.1915,18.Aug.=Nr. 241',
   '3.1913,15.Jan.=Nr. 106',
   '3.1913,29.Jan.=Nr. 108',
   '6.1916,14.Jun.=Nr. 284',
   '5.1915,21.Apr.=Nr. 224',
   'not_identified',
   'not_identified',
   '4.1914,14.Jan.=Nr. 158',
   '4.1914,4.Nov.=Nr. 200',
   '6.1916,11.Okt.=Nr. 301',
   '4.1914,3.Jun.=Nr. 178',
   '3.1913,1.Okt.=Nr. 143',
   '4.1914,28.Jan.=Nr. 160',
   '4.1914,14.Okt.=Nr. 197',
   '2.1912,27.Nov.=Nr. 99',
   '4.1914,21.Jan.=Nr. 159',
   '3.1913,12.Nov.=Nr. 149',
   'not_identified',
   '5.1915,17.Febr.=Nr. 215',
   '4.1914,18.Mrz.=Nr. 167',
   '3.1913,30.Jul.=Nr. 134',
   '4.1914,19.Aug.=Nr. 189',
   'not_identified',
   '5.1915,6.Jan.=Nr. 209',
   '5.1915,7.Apr.=Nr. 222',
   '3.1913,3.Sept.=Nr. 139',
   '5.1915,17.Mrz.=Nr. 219',
   '6.1916,12.Apr.=Nr. 275',
   '4.1914,25.Nov.=Nr. 203',
   'not_identified',
   '3.1913,22.Jan.=Nr. 107',
   '6.1916,20.Sept.=Nr. 298',
   '4.1914,2.Sept.=Nr. 191',
   '3.1913,22.Okt.=Nr. 146',
   '4.1914,29.Jul.=Nr. 186',
   'not_identified',
   '6.1916,5.Apr.=Nr. 274',
   '4.1914,8.Jul.=Nr. 183',
   '4.1914,15.Jul.=Nr. 184',
   'not_identified',
   '6.1916,19.Jan.=Nr. 263',
   '5.1915,24.Febr.=Nr. 216',
   '5.1915,20.Okt.=Nr. 250',
   '4.1914,11.Nov.=Nr. 201',
   '6.1916,9.Aug.=Nr. 292',
   '4.1914,18.Nov.=Nr. 202',
   '6.1916,1.Mrz.=Nr. 269',
   '3.1913,23.Jul.=Nr. 133',
   '5.1915,20.Jan.=Nr. 211',
   '4.1914,23.Dez.=Nr. 207',
   '5.1915,15.Sept.=Nr. 245',
   '6.1916,8.Mrz.=Nr. 270',
   '5.1915,3.Nov.=Nr. 252',
   '3.1913,16.Jul.=Nr. 132',
   '6.1916,23.Febr.=Nr. 268',
   '5.1915,29.Dez.=Nr. 260',
   'not_identified',
   '6.1916,15.Mrz.=Nr. 271',
   '4.1914,22.Jul.=Nr. 185',
   '3.1913,18.Jun.=Nr. 128',
   '6.1916,9.Febr.=Nr. 266',
   '4.1914,30.Dez.=Nr. 208',
   '6.1916,31.Mai=Nr. 282',
   '5.1915,28.Apr.=Nr. 225',
   '2.1912,21.Aug.=Nr. 85',
   '4.1914,18.Febr.=Nr. 163',
   '5.1915,23.Jun.=Nr. 233',
   'not_identified',
   '6.1916,6.Sept.=Nr. 296',
   '3.1913,2.Jul.=Nr. 130',
   '4.1914,21.Okt.=Nr. 198',
   '3.1913,29.Okt.=Nr. 147',
   'not_identified',
   '6.1916,17.Mai=Nr. 280',
   '6.1916,25.Okt.=Nr. 303',
   '2.1912,2.Okt.=Nr. 91',
   '2.1912,7.Aug.=Nr. 83',
   '3.1913,27.Aug.=Nr. 138',
   '4.1914,24.Jun.=Nr. 181'],
  'date': ['not_identified',
   '2.1912,11.Dez.=Nr. 101',
   '5.1915,7.Jul.=Nr. 235',
   '2.1912,11.Dez.=Nr. 101',
   '5.1915,7.Jul.=Nr. 235',
   '5.1915,31.Mrz.=Nr. 221',
   '5.1915,10.Mrz.=Nr. 218',
   '6.1916,16.Febr.=Nr. 267',
   'not_identified',
   '3.1913,25.Jun.=Nr. 129',
   '4.1914,23.Sept.=Nr. 194',
   'not_identified',
   '5.1915,18.Aug.=Nr. 241',
   '3.1913,15.Jan.=Nr. 106',
   '3.1913,29.Jan.=Nr. 108',
   '6.1916,14.Jun.=Nr. 284',
   '5.1915,21.Apr.=Nr. 224',
   'not_identified',
   'not_identified',
   '4.1914,14.Jan.=Nr. 158',
   '4.1914,4.Nov.=Nr. 200',
   '6.1916,11.Okt.=Nr. 301',
   '4.1914,3.Jun.=Nr. 178',
   '3.1913,1.Okt.=Nr. 143',
   '4.1914,28.Jan.=Nr. 160',
   '4.1914,14.Okt.=Nr. 197',
   '2.1912,27.Nov.=Nr. 99',
   '4.1914,21.Jan.=Nr. 159',
   '3.1913,12.Nov.=Nr. 149',
   'not_identified',
   '5.1915,17.Febr.=Nr. 215',
   '4.1914,18.Mrz.=Nr. 167',
   '3.1913,30.Jul.=Nr. 134',
   '4.1914,19.Aug.=Nr. 189',
   'not_identified',
   '5.1915,6.Jan.=Nr. 209',
   '5.1915,7.Apr.=Nr. 222',
   '3.1913,3.Sept.=Nr. 139',
   '5.1915,17.Mrz.=Nr. 219',
   '6.1916,12.Apr.=Nr. 275',
   '4.1914,25.Nov.=Nr. 203',
   'not_identified',
   '3.1913,22.Jan.=Nr. 107',
   '6.1916,20.Sept.=Nr. 298',
   '4.1914,2.Sept.=Nr. 191',
   '3.1913,22.Okt.=Nr. 146',
   '4.1914,29.Jul.=Nr. 186',
   'not_identified',
   '6.1916,5.Apr.=Nr. 274',
   '4.1914,8.Jul.=Nr. 183',
   '4.1914,15.Jul.=Nr. 184',
   'not_identified',
   '6.1916,19.Jan.=Nr. 263',
   '5.1915,24.Febr.=Nr. 216',
   '5.1915,20.Okt.=Nr. 250',
   '4.1914,11.Nov.=Nr. 201',
   '6.1916,9.Aug.=Nr. 292',
   '4.1914,18.Nov.=Nr. 202',
   '6.1916,1.Mrz.=Nr. 269',
   '3.1913,23.Jul.=Nr. 133',
   '5.1915,20.Jan.=Nr. 211',
   '4.1914,23.Dez.=Nr. 207',
   '5.1915,15.Sept.=Nr. 245',
   '6.1916,8.Mrz.=Nr. 270',
   '5.1915,3.Nov.=Nr. 252',
   '3.1913,16.Jul.=Nr. 132',
   '6.1916,23.Febr.=Nr. 268',
   '5.1915,29.Dez.=Nr. 260',
   'not_identified',
   '6.1916,15.Mrz.=Nr. 271',
   '4.1914,22.Jul.=Nr. 185',
   '3.1913,18.Jun.=Nr. 128',
   '6.1916,9.Febr.=Nr. 266',
   '4.1914,30.Dez.=Nr. 208',
   '6.1916,31.Mai=Nr. 282',
   '5.1915,28.Apr.=Nr. 225',
   '2.1912,21.Aug.=Nr. 85',
   '4.1914,18.Febr.=Nr. 163',
   '5.1915,23.Jun.=Nr. 233',
   'not_identified',
   '6.1916,6.Sept.=Nr. 296',
   '3.1913,2.Jul.=Nr. 130',
   '4.1914,21.Okt.=Nr. 198',
   '3.1913,29.Okt.=Nr. 147',
   'not_identified',
   '6.1916,17.Mai=Nr. 280',
   '6.1916,25.Okt.=Nr. 303',
   '2.1912,2.Okt.=Nr. 91',
   '2.1912,7.Aug.=Nr. 83',
   '3.1913,27.Aug.=Nr. 138',
   '4.1914,24.Jun.=Nr. 181'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q2748457',
  'wikidata_label': 'Ricardo de la Cierva',
  'wikidata_description': 'Spanish historian and politician (1926–2015)',
  'match_confidence': 0.767,
  'matched_name': 'Heliográficos de Ricardo'},
 {'author_name': 'Hector Gómez',
  'initial_names': ['Hector Gómez',
   'Victor Gómez',
   'Héctor Gómez',
   'Victor Gómez',
   'Hector Gómez'],
  'journal_index': ['980', '436', '4591', '753', '2231'],
  'journal_name': ['not_identified_980',
   'La-Literatura-Argentina-72',
   'not_identified_4591',
   'HCS-115',
   'not_identified_2231'],
  'issue': ['not_identified',
   'La-Literatura-Argentina-72',
   'not_identified',
   'HCS-115',
   'not_identified'],
  'date': ['not_identified',
   'agosto de 1934',
   'not_identified',
   '18 de febrero de 1959',
   'not_identified'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q89905165',
  'wikidata_label': 'Hector Gomez',
  'wikidata_description': 'researcher',
  'match_confidence': 1.0,
  'matched_name': 'Hector Gómez'},
 {'author_name': 'Carlos Se',
  'initial_names': ['Cas Se',
   'Cas Es',
   'Ads Se',
   'Cra Se',
   'Cas re',
   'Jas Se',
   'Carlos Se',
   'Cas De',
   'Ca Se',
   'Ca Se'],
  'journal_index': ['980',
   2189,
   3682,
   1633,
   2388,
   3275,
   875,
   1721,
   '3891',
   1988],
  'journal_name': ['not_identified_980',
   'Atlántida',
   'Vida montevideana',
   'Fray mocho',
   'Mundo argentino',
   'Céltiga',
   'Mundo argentino',
   'Fray mocho',
   'not_identified_3891',
   'Mundo argentino'],
  'issue': ['not_identified',
   '4.1921,26.Mai=Nr. 165',
   '2.1898,16.Jan.=Nr. 29',
   '14.1925,1.Dez.=Nr. 710',
   '3.1913,1.Okt.=Nr. 143',
   '5.1928=Nr. 78',
   '9.1919,20.Aug.=Nr. 449',
   '18.1929,18.Jun.=Nr. 895',
   'not_identified',
   '4.1914,1.Apr.=Nr. 169'],
  'date': ['not_identified',
   '4.1921,26.Mai=Nr. 165',
   '2.1898,16.Jan.=Nr. 29',
   '14.1925,1.Dez.=Nr. 710',
   '3.1913,1.Okt.=Nr. 143',
   '5.1928=Nr. 78',
   '9.1919,20.Aug.=Nr. 449',
   '18.1929,18.Jun.=Nr. 895',
   'not_identified',
   '4.1914,1.Apr.=Nr. 169'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q3753357',
  'wikidata_label': 'Carlos Seco Serrano',
  'wikidata_description': 'Spanish historian (1923–2020)',
  'match_confidence': 0.9,
  'matched_name': 'Carlos Se'},
 {'author_name': 'Dejo Morales',
  'initial_names': ['Eo Moral',
   'Mateo Moral',
   'Eos Mol',
   'Es Mora',
   'Ea Mora',
   'Dejo Morales'],
  'journal_index': ['980', '3707', '67', '750', 4094, 3529],
  'journal_name': ['not_identified_980',
   'not_identified_3707',
   'Mas-Alla-30',
   'rmn28',
   'Mundo argentino',
   'Mundo argentino'],
  'issue': ['not_identified',
   'not_identified',
   'Mas-Alla-30',
   'rmn28',
   '8.1918,16.Okt.=Nr. 406',
   '6.1916,26.Apr.=Nr. 277'],
  'date': ['not_identified',
   'not_identified',
   'Noviembre de 1955',
   'Febrero de 1934',
   '8.1918,16.Okt.=Nr. 406',
   '6.1916,26.Apr.=Nr. 277'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q64848964',
  'wikidata_label': 'Mateo Morales',
  'wikidata_description': 'actor',
  'match_confidence': 1.0,
  'matched_name': 'Mateo Moral'},
 {'author_name': 'Ricardo Vegas Garcia',
  'initial_names': ['Ricardo Garcia',
   'Ricardo Uri',
   'Ricardo Uri',
   'Ricardo Garde',
   'Ricardo Paiva',
   'Ricardo García',
   'Ricardo Cal',
   'Ricardo Ra',
   'Ricardo Car',
   'Ricardo Gan',
   'Ricardo Aparicio',
   'Ricardo Vegas Garcia',
   'Ricardo Riva',
   'Ricardo Mariani',
   'Ricardo Gadea',
   'Ricardo Scarica'],
  'journal_index': ['980',
   '4502',
   '4502',
   '271',
   2594,
   2586,
   3052,
   2535,
   2520,
   1525,
   949,
   2409,
   1538,
   '1554',
   3303,
   '578'],
  'journal_name': ['not_identified_980',
   'not_identified_4502',
   'not_identified_4502',
   'Ciudad-2',
   'Fray mocho',
   'Mundo argentino',
   'La nota',
   'Mundo argentino',
   'Céltiga',
   'Fray mocho',
   'El gladiador',
   'Mercurio peruano',
   'Correo musical sud-americano',
   'not_identified_1554',
   'Mundo argentino',
   'mdn09_b'],
  'issue': ['not_identified',
   'not_identified',
   'not_identified',
   'Ciudad-2',
   '8.1919,14.Okt.=Nr. 390',
   '1.1911,18.Okt.=Nr. 41',
   '4.1919,14.März=Nr. 188',
   '12.1922,2.Aug.=Nr. 602',
   '5.1928=Nr. 81',
   '15.1926,2.Mrz.=Nr. 723',
   '2.1902,30.Mai=Nr. 26',
   'Año 3.1920=Nr. 25',
   '3.1917,20.Jun.=Nr. 117',
   'not_identified',
   '10.1920,13.Okt.=Nr. 509',
   'mdn09_b'],
  'date': ['not_identified',
   'not_identified',
   'not_identified',
   'Segundo y tercer trimestre de 1955',
   '8.1919,14.Okt.=Nr. 390',
   '1.1911,18.Okt.=Nr. 41',
   '4.1919,14.März=Nr. 188',
   '12.1922,2.Aug.=Nr. 602',
   '5.1928=Nr. 81',
   '15.1926,2.Mrz.=Nr. 723',
   '2.1902,30.Mai=Nr. 26',
   'Año 3.1920=Nr. 25',
   '3.1917,20.Jun.=Nr. 117',
   'not_identified',
   '10.1920,13.Okt.=Nr. 509',
   'Octubre de 1959'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q18419011',
  'wikidata_label': 'Ricardo Vegas García',
  'wikidata_description': 'Peruvian journalist and diplomat',
  'match_confidence': 1.0,
  'matched_name': 'Ricardo Vegas Garcia'},
 {'author_name': 'Myrian Harry',
  'initial_names': ['Myrian Harry', 'Myriam Harry'],
  'journal_index': [2696, 3327],
  'journal_name': ['Atlántida', 'Cuba contemporánea'],
  'issue': ['10.1927,29.Dez.=Nr. 507', 'T. 30.1922,118'],
  'date': ['10.1927,29.Dez.=Nr. 507', 'T. 30.1922,118'],
  'is_human': True,
  'gender': None,
  'country': None,
  'occupation': None,
  'language': None,
  'genre': None,
  'dob': None,
  'dod': None,
  'wikidata_id': 'Q3331525',
  'wikidata_label': 'Myriam Harry',
  'wikidata_description': 'pen name of Maria Rosette Shapira (April 1869 (some sources say 1875) – March 10, 1958), a French journalist and writer',
  'match_confidence': 1.0,
  'matched_name': 'Myriam Harry'}]

In [6]:

import pickle

with open('ahira_cmola_toc_contributors_by_wiki_via_distilgpt2_n_phi2_resolved_list.pkl', 'wb') as fp:
                    pickle.dump(resolved_list, fp)

fp.close()

In [7]:
type(resolved_list)

list

In [3]:

import json

In [4]:
with open("ahira_cmola_wikidata_metadata_cache.json", "r") as file:
    wikidata_cache = json.load(file)

In [8]:


# 1. Build a set of all wikidata_ids in the resolved list
ids_to_remove = {item["wikidata_id"] for item in resolved_list if item.get("wikidata_id")}

# 2. Remove those keys from the big JSON dict
wiki_cache_unresolved = {
    qid: info
    for qid, info in wikidata_cache.items()
    if qid not in ids_to_remove
}

# (Optional) Save the result back to JSON
import json
with open("ahira_cmola_toc_wiki_cache_unresolved.json", "w") as f:
    json.dump(wiki_cache_unresolved, f, indent=2)


In [10]:

with open("ahira_cmola_toc_collapsed_by_phi-2_and_rapidfuzz_final.json", "r", encoding="utf-8") as f:
    contributors_cleaned_by_phi2 = json.load(f)

f.close()

In [15]:

len(wiki_cache_unresolved)

11155

In [ ]:

# SINCE THE WIKI SEARCHES INOLVED 'COUNTRY' AND 'OCCUPATION' WHICH OFTEN (OR ALWAYS?) ARE ENTERED IN THE PLURAL, SOME NON-FINDS WERE ACTUALLY FINDS
# SO BELOW WE ARE ITERATING THRU THE WIKIDATA SEARCH RESULT CACHE AND ADDING SUCH FALSE NON-FINDS TO THE FINDS LIST


In [14]:


from datetime import datetime

# -------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------


YEAR_LIMIT = 1948
PICKLE_EVERY = 100
PICKLE_PATH = "ahira_cmola_toc_contributors_wikidata_resolved_list.pkl"
UNMATCHED_JSON_PATH = "ahira_cmola_toc_unmatched_wikidata.json"

# -------------------------------------------------------------
# HELPER: Check DOB
# -------------------------------------------------------------

def dob_before_limit(dob, year_limit=YEAR_LIMIT):
    """Return True if dob is None or dob < year_limit."""
    if dob is None or dob == "null":
        return True
    try:
        year = int(dob[:4])  # Wikidata DOBs begin with YYYY
        return year < year_limit
    except:
        return False

# -------------------------------------------------------------
# MAIN PROCESS
# -------------------------------------------------------------

# resolved_list = [] # NO, WE ALREADY GOT a resolved_list!!!!!!!!!
unmatched_entries = {}   # key = wikidata_id, value = Wikidata record

count = 0

for qid, wd in wiki_cache_unresolved.items():

    # ---------------------------------------------------------
    # 1. Eligibility check (occupation + dob)
    # ---------------------------------------------------------
    occupations = wd.get("occupations") or []

    if not any(occ in DOMAIN_KEYWORDS for occ in occupations):
        unmatched_entries[qid] = wd
        continue

    if not dob_before_limit(wd.get("dob")):
        unmatched_entries[qid] = wd
        continue

    # ---------------------------------------------------------
    # 2. Exact name matching (labels → contributor)
    # ---------------------------------------------------------
    labels = wd.get("labels", [])
    if not labels:
        unmatched_entries[qid] = wd
        continue

    matched_contributor = None

    for label in wd.get("labels", []):
        for c in contributors_cleaned_by_phi2:
            if (
                label == c.get("author_name") or
                label in c.get("initial_names", [])
            ):
                matched_contributor = c
                break
        if matched_contributor:
            break


    if matched_contributor is None:
        unmatched_entries[qid] = wd
        continue

    # ---------------------------------------------------------
    # 3. Build merged, enriched entry
    # ---------------------------------------------------------
    merged = {
        # original contributor data
        "author_name": matched_contributor.get("author_name"),
        "initial_names": matched_contributor.get("initial_names"),
        "journal_index": matched_contributor.get("journal_index"),
        "journal_name": matched_contributor.get("journal_name"),
        "issue": matched_contributor.get("issue"),
        "date": matched_contributor.get("date"),

        # wikidata fields
        "is_human": wd.get("is_human"),
        "gender": wd.get("gender"),
        "countries": wd.get("countries"),
        "occupations": wd.get("occupations"),
        "language": wd.get("language"),
        "languages_written": wd.get("languages_written"),
        "genres": wd.get("genres"),
        "dob": wd.get("dob"),
        "dod": wd.get("dod"),
        "wikidata_id": qid,
        "labels": wd.get("labels"),
        "wikidata_description": wd.get("description") if "description" in wd else None,

        # match metadata (if present)
        "match_confidence": wd.get("match_confidence") if "match_confidence" in wd else None,
        "matched_name": wd.get("matched_name") if "matched_name" in wd else None
    }

    resolved_list.append(merged)
    count += 1

    # ---------------------------------------------------------
    # 4. Periodic pickling
    # ---------------------------------------------------------
    if count % PICKLE_EVERY == 0:
        with open(PICKLE_PATH, "wb") as f:
            pickle.dump(resolved_list, f)
        print(f"[INFO] Pickled {len(resolved_list)} entries so far.")

# ---------------------------------------------------------
# FINAL SAVES
# ---------------------------------------------------------

# save resolved list
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(resolved_list, f)

# save unmatched entries to JSON
with open(UNMATCHED_JSON_PATH, "w") as f:
    json.dump(unmatched_entries, f, indent=2)

print(f"[DONE] Resolved: {len(resolved_list)}, Unmatched: {len(unmatched_entries)}")


[INFO] Pickled 108 entries so far.
[INFO] Pickled 208 entries so far.
[INFO] Pickled 308 entries so far.
[INFO] Pickled 408 entries so far.
[INFO] Pickled 508 entries so far.
[INFO] Pickled 608 entries so far.
[INFO] Pickled 708 entries so far.
[INFO] Pickled 808 entries so far.
[INFO] Pickled 908 entries so far.
[INFO] Pickled 1008 entries so far.
[INFO] Pickled 1108 entries so far.
[INFO] Pickled 1208 entries so far.
[INFO] Pickled 1308 entries so far.
[INFO] Pickled 1408 entries so far.
[INFO] Pickled 1508 entries so far.
[INFO] Pickled 1608 entries so far.
[INFO] Pickled 1708 entries so far.
[INFO] Pickled 1808 entries so far.
[INFO] Pickled 1908 entries so far.
[INFO] Pickled 2008 entries so far.
[INFO] Pickled 2108 entries so far.
[INFO] Pickled 2208 entries so far.
[INFO] Pickled 2308 entries so far.
[INFO] Pickled 2408 entries so far.
[INFO] Pickled 2508 entries so far.
[INFO] Pickled 2608 entries so far.
[INFO] Pickled 2708 entries so far.
[INFO] Pickled 2808 entries so far.
[

In [17]:

len(resolved_list)

4011

In [19]:
len(contributors_cleaned_by_phi2)

63544

In [ ]:

# SO RESOLVED_LIST CONTAINS 4,011 POTENTIAL CONTRIBUTORS FOUND ON WIKIDATA OUT OF THE 'INITAL' 63,544 (CLEANED AND COLLAPSED FROM THE ACTUALLY INITIAL OVER HALF MILLION)


In [22]:

latin_american_countries_and_spain = {
        "argentina", "bolivia", "chile", "colombia", "costa rica", "cuba",
        "dominican republic", "ecuador", "el salvador", "guatemala", "gran colombia",
        "haiti", "honduras", "mexico", "nicaragua", "panama", "paraguay", "peru",
        "uruguay", "venezuela", "puerto rico", "spain", "brazil",
        "new spain", "virreinato del peru", "colonial mexico"
    }

In [ ]:

# THESE ARE OUR FINAL FINDS ('FINAL' as in, for now, and before further refinement & exploration)


In [24]:

from collections import defaultdict


originals = 0
translations = 0
translations_by_country = defaultdict(int)

for entry in resolved_list:
    num_journals = len(entry.get("journal_index", []))
    countries = entry.get("countries") or []

    countries_lower = [c.lower() for c in countries]

    if any(c in latin_american_countries_and_spain for c in countries_lower):
        originals += num_journals
    else:
        translations += num_journals
        for c in countries_lower:
            translations_by_country[c] += num_journals


total = originals + translations
translations_pct = (translations / total) * 100 if total > 0 else 0


translations_country_stats = {}
for c, count in translations_by_country.items():
    translations_country_stats[c] = {
        "count": count,
        "percentage": (count / translations) * 100 if translations > 0 else 0
    }

# Display results
print(f"Originals: {originals}")
print(f"Translations: {translations}")
print(f"Percentage of translations: {translations_pct:.2f}%")
print("Translations by country:")
for c, stats in translations_country_stats.items():
    print(f"  {c}: {stats['count']} ({stats['percentage']:.2f}%)")


Originals: 15650
Translations: 9751
Percentage of translations: 38.39%
Translations by country:
  kingdom of italy: 396 (4.06%)
  italy: 299 (3.07%)
  united states: 2213 (22.70%)
  france: 2247 (23.04%)
  united kingdom: 535 (5.49%)
  united kingdom of great britain and ireland: 390 (4.00%)
  switzerland: 52 (0.53%)
  russian empire: 52 (0.53%)
  kingdom of france: 36 (0.37%)
  germany: 243 (2.49%)
  austria: 106 (1.09%)
  kingdom of prussia: 22 (0.23%)
  german empire: 38 (0.39%)
  kingdom of england: 50 (0.51%)
  hungary: 40 (0.41%)
  ottoman empire: 108 (1.11%)
  belgium: 73 (0.75%)
  canada: 180 (1.85%)
  sweden: 69 (0.71%)
  classical athens: 2 (0.02%)
  ireland: 85 (0.87%)
  kingdom of sardinia: 13 (0.13%)
  crown of castile: 84 (0.86%)
  kingdom of bavaria: 9 (0.09%)
  kingdom of portugal: 40 (0.41%)
  kingdom of great britain: 106 (1.09%)
  seljuk empire: 5 (0.05%)
  german reich: 59 (0.61%)
  spanish empire: 13 (0.13%)
  romania: 13 (0.13%)
  kingdom of the netherlands: 33 (0

In [25]:

from collections import defaultdict


originals = 0
translations = 0

originals_by_country = defaultdict(int)
translations_by_country = defaultdict(int)

for entry in resolved_list:
    num_journals = len(entry.get("journal_index", []))
    countries = entry.get("countries") or []
    countries_lower = [c.lower() for c in countries]

    # Check if any country is in Latin America/Spain
    if any(c in latin_american_countries_and_spain for c in countries_lower):
        originals += num_journals
        for c in countries_lower:
            if c in latin_american_countries_and_spain:
                originals_by_country[c] += num_journals
    else:
        translations += num_journals
        for c in countries_lower:
            translations_by_country[c] += num_journals


total = originals + translations
translations_pct = (translations / total) * 100 if total > 0 else 0


def compute_country_stats(country_dict, total_contributions):
    stats = {}
    for c, count in country_dict.items():
        stats[c] = {
            "count": count,
            "percentage": (count / total_contributions) * 100 if total_contributions > 0 else 0
        }
    # Sort descending by count
    stats_sorted = dict(sorted(stats.items(), key=lambda x: x[1]['count'], reverse=True))
    return stats_sorted

originals_country_stats = compute_country_stats(originals_by_country, originals)
translations_country_stats = compute_country_stats(translations_by_country, translations)


print(f"Originals: {originals}")
print(f"Translations: {translations}")
print(f"Percentage of translations: {translations_pct:.2f}%\n")

print("Originals by country (descending):")
for c, stats in originals_country_stats.items():
    print(f"  {c}: {stats['count']} ({stats['percentage']:.2f}%)")

print("\nTranslations by country (descending):")
for c, stats in translations_country_stats.items():
    print(f"  {c}: {stats['count']} ({stats['percentage']:.2f}%)")


Originals: 15650
Translations: 9751
Percentage of translations: 38.39%

Originals by country (descending):
  argentina: 7804 (49.87%)
  spain: 4044 (25.84%)
  uruguay: 1321 (8.44%)
  peru: 1044 (6.67%)
  mexico: 541 (3.46%)
  chile: 417 (2.66%)
  colombia: 404 (2.58%)
  cuba: 291 (1.86%)
  brazil: 274 (1.75%)
  nicaragua: 152 (0.97%)
  venezuela: 129 (0.82%)
  dominican republic: 94 (0.60%)
  paraguay: 90 (0.58%)
  bolivia: 55 (0.35%)
  honduras: 24 (0.15%)
  ecuador: 23 (0.15%)
  guatemala: 22 (0.14%)
  costa rica: 11 (0.07%)
  el salvador: 5 (0.03%)
  panama: 1 (0.01%)

Translations by country (descending):
  france: 2247 (23.04%)
  united states: 2213 (22.70%)
  united kingdom: 535 (5.49%)
  kingdom of italy: 396 (4.06%)
  united kingdom of great britain and ireland: 390 (4.00%)
  italy: 299 (3.07%)
  germany: 243 (2.49%)
  canada: 180 (1.85%)
  ottoman empire: 108 (1.11%)
  austria: 106 (1.09%)
  kingdom of great britain: 106 (1.09%)
  ancient rome: 91 (0.93%)
  ireland: 85 (0.87%)

In [27]:


with open("ahira_cmola_toc_originals_country_stats.json", "w") as f:
    json.dump(originals_country_stats, f, indent=2)

# Save translations by country
with open("ahira_cmola_toc_translations_country_stats.json", "w") as f:
    json.dump(translations_country_stats, f, indent=2)


In [26]:

non_latin_authors_list = []

for entry in resolved_list:
    countries = entry.get("countries") or []
    countries_lower = [c.lower() for c in countries]

    if not any(c in latin_american_countries_and_spain for c in countries_lower):
        non_latin_authors_list.append({
            "author_name": entry.get("author_name"),
            "initial_names": entry.get("initial_names"),
            "countries": countries_lower
        })

# Save to JSON
with open("ahira_cmola_toc_translated_authors.json", "w") as f:
    json.dump(non_latin_authors_list, f, indent=2)

print(f"Saved {len(non_latin_authors_list)} non-Latin American/Spain authors to 'ahira_cmola_toc_translated_authors.json'.")


Saved 2106 non-Latin American/Spain authors to 'ahira_cmola_toc_translated_authors.json'.
